# Workshop: Bias di Genere nei Dataset Clinici
## Fase 3: Performance Gap tra Sottogruppi
## Fase 4: Il Bias Nasce dal Dataset

---

**Dataset**: GOSSIS (Global Open Source Severity of Illness Score)  
**Task**: Predizione della mortalita' ospedaliera in terapia intensiva (ICU)  
**Target**: `hospital_death` (0 = sopravvissuto, 1 = deceduto)  

**Obiettivo di questo notebook**:
1. Addestrare 3 modelli di complessita' crescente e verificare se il **gap di performance tra generi** esiste
2. Dimostrare che il bias **nasce dal dataset**, non dall'algoritmo

---

## Sezione 1: Setup e Caricamento Dati

In questa sezione installiamo le librerie necessarie, montiamo Google Drive e carichiamo i dati gia' preprocessati.

In [ ]:
# Montaggio Google Drive (eseguire solo su Google Colab)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Installazione librerie necessarie
!pip install xgboost -q

In [ ]:
# Importazione librerie
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from xgboost import XGBClassifier
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings('ignore')

# Stile dei grafici
sns.set_style('whitegrid')

# Colori coerenti: blu per maschi, arancione/rosso per femmine
COLOR_M = '#1f77b4'  # blu
COLOR_F = '#e74c3c'  # rosso/arancione
COLOR_OVERALL = '#2ca02c'  # verde per overall

print('Librerie caricate con successo!')

In [ ]:
# Percorso dei dati su Google Drive
# MODIFICA questo percorso in base alla tua struttura di cartelle
BASE_PATH = '/content/drive/MyDrive/Workshop_1/Workshop_Dataset/'

# Caricamento dei dati preprocessati
train_features = pd.read_csv(BASE_PATH + 'train_features.csv')
train_labels = pd.read_csv(BASE_PATH + 'train_labels.csv')
test_features = pd.read_csv(BASE_PATH + 'test_features.csv')
test_labels = pd.read_csv(BASE_PATH + 'test_labels.csv')

print('Dati caricati con successo!')

In [ ]:
# Controllo rapido delle dimensioni
print(f'Training set:  {train_features.shape[0]} righe, {train_features.shape[1]} features')
print(f'Test set:      {test_features.shape[0]} righe, {test_features.shape[1]} features')
print(f'Target train:  {train_labels.shape}')
print(f'Target test:   {test_labels.shape}')
print()
print('Prime 5 colonne:', list(train_features.columns[:5]))
print('Ultime 5 colonne:', list(train_features.columns[-5:]))

In [ ]:
# Prepariamo i vettori di target come array 1D
y_train = train_labels.values.ravel()
y_test = test_labels.values.ravel()

print(f'Tasso di mortalita nel training: {y_train.mean():.3f} ({y_train.mean()*100:.1f}%)')
print(f'Tasso di mortalita nel test:     {y_test.mean():.3f} ({y_test.mean()*100:.1f}%)')

### Funzione di Valutazione per Sottogruppi

Questa funzione e' il cuore dell'analisi: valuta un modello **separatamente** per maschi e femmine, oltre che sul dataset complessivo.

In [ ]:
def evaluate_model_subgroups(model, X, y, model_name='Modello', is_pytorch=False):
    """
    Valuta un modello su Overall, Maschi e Femmine separatamente.
    
    Parametri:
    - model: modello addestrato (sklearn, xgboost o pytorch)
    - X: DataFrame delle features
    - y: array dei target
    - model_name: nome del modello per la tabella
    - is_pytorch: True se il modello e' PyTorch
    
    Restituisce:
    - DataFrame con metriche per ogni sottogruppo
    """
    # Funzione interna per ottenere predizioni
    def get_predictions(model, X_sub, is_pytorch):
        if is_pytorch:
            model.eval()
            with torch.no_grad():
                X_tensor = torch.FloatTensor(X_sub.values)
                y_prob = model(X_tensor).numpy().ravel()
            y_pred = (y_prob >= 0.5).astype(int)
        else:
            y_pred = model.predict(X_sub)
            y_prob = model.predict_proba(X_sub)[:, 1]
        return y_pred, y_prob
    
    # Funzione interna per calcolare le metriche
    def compute_metrics(y_true, y_pred, y_prob):
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        return {
            'N campioni': len(y_true),
            'Accuracy': accuracy_score(y_true, y_pred),
            'Precision': precision_score(y_true, y_pred, zero_division=0),
            'Recall (Sensitivity)': recall_score(y_true, y_pred, zero_division=0),
            'Specificity': specificity,
            'F1': f1_score(y_true, y_pred, zero_division=0),
            'AUC-ROC': roc_auc_score(y_true, y_prob)
        }
    
    # Maschere per genere
    mask_M = X['gender_M'] == 1 if 'gender_M' in X.columns else X['gender_M'] > 0
    mask_F = X['gender_F'] == 1 if 'gender_F' in X.columns else X['gender_F'] > 0
    
    results = {}
    
    # Overall
    y_pred, y_prob = get_predictions(model, X, is_pytorch)
    results['Overall'] = compute_metrics(y, y_pred, y_prob)
    
    # Maschi
    y_pred_m, y_prob_m = get_predictions(model, X[mask_M], is_pytorch)
    results['Maschi'] = compute_metrics(y[mask_M], y_pred_m, y_prob_m)
    
    # Femmine
    y_pred_f, y_prob_f = get_predictions(model, X[mask_F], is_pytorch)
    results['Femmine'] = compute_metrics(y[mask_F], y_pred_f, y_prob_f)
    
    df = pd.DataFrame(results).T
    df.index.name = model_name
    
    # Formattazione
    df['N campioni'] = df['N campioni'].astype(int)
    for col in df.columns:
        if col != 'N campioni':
            df[col] = df[col].round(4)
    
    return df

print('Funzione evaluate_model_subgroups definita.')

---

## Sezione 2: Modello 1 — Regressione Logistica (Baseline)

La **Regressione Logistica** e' il modello piu' semplice per la classificazione binaria.  
E' un modello **lineare**: traccia una linea retta per separare i pazienti che sopravvivono da quelli che non sopravvivono.  
E' veloce da addestrare e facile da interpretare. Lo usiamo come **baseline** (punto di riferimento).

In [ ]:
# Addestramento della Regressione Logistica
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(train_features, y_train)

print('Regressione Logistica addestrata!')

In [ ]:
# Valutazione sul test set: Overall + per genere
lr_results = evaluate_model_subgroups(lr_model, test_features, y_test, 'Logistic Regression')
print('=== REGRESSIONE LOGISTICA — Risultati per Sottogruppo ===')
print()
lr_results

In [ ]:
# Grafico a barre: confronto metriche Maschi vs Femmine
metrics_to_plot = ['AUC-ROC', 'Recall (Sensitivity)', 'Specificity', 'Precision', 'F1']

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metrics_to_plot))
width = 0.35

bars_m = ax.bar(x - width/2, lr_results.loc['Maschi', metrics_to_plot],
                width, label='Maschi', color=COLOR_M, alpha=0.85)
bars_f = ax.bar(x + width/2, lr_results.loc['Femmine', metrics_to_plot],
                width, label='Femmine', color=COLOR_F, alpha=0.85)

ax.set_ylabel('Valore')
ax.set_title('Regressione Logistica — Confronto Maschi vs Femmine')
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot, rotation=15)
ax.legend()
ax.set_ylim(0, 1)

# Aggiungiamo i valori sulle barre
for bar in bars_m:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars_f:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Matrici di confusione per Maschi e Femmine (affiancate)
mask_M_test = test_features['gender_M'] == 1
mask_F_test = test_features['gender_F'] == 1

y_pred_lr = lr_model.predict(test_features)

cm_m = confusion_matrix(y_test[mask_M_test], y_pred_lr[mask_M_test])
cm_f = confusion_matrix(y_test[mask_F_test], y_pred_lr[mask_F_test])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(cm_m, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Pred: Vivo', 'Pred: Deceduto'],
            yticklabels=['Reale: Vivo', 'Reale: Deceduto'])
axes[0].set_title('Maschi — Matrice di Confusione (LR)')

sns.heatmap(cm_f, annot=True, fmt='d', cmap='Reds', ax=axes[1],
            xticklabels=['Pred: Vivo', 'Pred: Deceduto'],
            yticklabels=['Reale: Vivo', 'Reale: Deceduto'])
axes[1].set_title('Femmine — Matrice di Confusione (LR)')

plt.tight_layout()
plt.show()

---

## Sezione 3: Modello 2 — XGBoost

**XGBoost** (Extreme Gradient Boosting) e' un modello molto piu' potente della Regressione Logistica.  
Funziona combinando centinaia di **alberi decisionali** (come un comitato di esperti che votano insieme).  
E' uno dei modelli piu' usati nelle competizioni di machine learning e in molte applicazioni cliniche.

In [ ]:
# Addestramento di XGBoost
xgb_model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    eval_metric='logloss'
)
xgb_model.fit(train_features, y_train)

print('XGBoost addestrato!')

In [ ]:
# Valutazione sul test set: Overall + per genere
xgb_results = evaluate_model_subgroups(xgb_model, test_features, y_test, 'XGBoost')
print('=== XGBOOST — Risultati per Sottogruppo ===')
print()
xgb_results

In [ ]:
# Grafico a barre: confronto metriche Maschi vs Femmine
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metrics_to_plot))
width = 0.35

bars_m = ax.bar(x - width/2, xgb_results.loc['Maschi', metrics_to_plot],
                width, label='Maschi', color=COLOR_M, alpha=0.85)
bars_f = ax.bar(x + width/2, xgb_results.loc['Femmine', metrics_to_plot],
                width, label='Femmine', color=COLOR_F, alpha=0.85)

ax.set_ylabel('Valore')
ax.set_title('XGBoost — Confronto Maschi vs Femmine')
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot, rotation=15)
ax.legend()
ax.set_ylim(0, 1)

for bar in bars_m:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars_f:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Confronto Regressione Logistica vs XGBoost (tabella affiancata)
comparison_lr_xgb = pd.concat(
    [lr_results, xgb_results],
    keys=['Logistic Regression', 'XGBoost'],
    names=['Modello', 'Sottogruppo']
)
print('=== CONFRONTO: Logistic Regression vs XGBoost ===')
print()
comparison_lr_xgb

---

## Sezione 4: Modello 3 — Mini-MLP (Deep Learning)

Una **rete neurale** (MLP = Multi-Layer Perceptron) e' l'architettura base del **deep learning**.  
Immaginate strati di neuroni artificiali che imparano relazioni complesse nei dati.  
Usiamo una rete molto semplice con 2 strati nascosti: e' gia' molto piu' potente dei modelli precedenti.

**Struttura della rete**:
- Input: 119 features
- Strato 1: 64 neuroni
- Strato 2: 32 neuroni  
- Output: 1 neurone (probabilita' di mortalita')

In [ ]:
# Definizione della rete neurale
class MiniMLP(nn.Module):
    def __init__(self, input_size=119):
        super(MiniMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.network(x)

print('Architettura della rete neurale definita.')

In [ ]:
# Preparazione dei dati per PyTorch
X_train_tensor = torch.FloatTensor(train_features.values)
y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1)
X_test_tensor = torch.FloatTensor(test_features.values)
y_test_tensor = torch.FloatTensor(y_test).unsqueeze(1)

# Dataset e DataLoader per il training
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True)

# Calcolo del peso per la classe positiva (per gestire lo sbilanciamento)
n_pos = y_train.sum()
n_neg = len(y_train) - n_pos
pos_weight = torch.FloatTensor([n_neg / n_pos])

print(f'Peso classe positiva: {pos_weight.item():.2f}')
print(f'(Ci sono {pos_weight.item():.1f} volte piu sopravvissuti che deceduti)')

In [ ]:
# Addestramento della rete neurale
input_size = train_features.shape[1]
mlp_model = MiniMLP(input_size=input_size)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
# Nota: usiamo BCEWithLogitsLoss, quindi modifichiamo il modello per non avere Sigmoid finale
# In alternativa usiamo BCELoss con il modello che ha gia' Sigmoid
criterion = nn.BCELoss()  # Il modello ha gia' Sigmoid nell'ultimo strato

optimizer = torch.optim.Adam(mlp_model.parameters(), lr=0.001)

# Per gestire lo sbilanciamento con BCELoss, usiamo pesi manuali nel loss
# Creiamo un weight tensor per ogni sample
pos_w = n_neg / n_pos

# Training loop
n_epochs = 10
losses = []

mlp_model.train()
for epoch in range(n_epochs):
    epoch_loss = 0
    n_batches = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_hat = mlp_model(X_batch)
        
        # Peso manuale: piu' peso ai campioni positivi (deceduti)
        weights = torch.where(y_batch == 1, pos_w, 1.0)
        loss = (nn.functional.binary_cross_entropy(y_hat, y_batch, reduction='none') * weights).mean()
        
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    
    avg_loss = epoch_loss / n_batches
    losses.append(avg_loss)
    
    if (epoch + 1) % 2 == 0:
        print(f'Epoca {epoch+1}/{n_epochs} - Loss: {avg_loss:.4f}')

print('\nRete neurale addestrata!')

In [ ]:
# Curva di apprendimento (training loss)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, n_epochs + 1), losses, color='purple', linewidth=2)
ax.set_xlabel('Epoca')
ax.set_ylabel('Loss')
ax.set_title('Mini-MLP — Curva di Apprendimento')
plt.tight_layout()
plt.show()

In [ ]:
# Valutazione della rete neurale sul test set
mlp_results = evaluate_model_subgroups(mlp_model, test_features, y_test, 'Mini-MLP', is_pytorch=True)
print('=== MINI-MLP (Deep Learning) — Risultati per Sottogruppo ===')
print()
mlp_results

In [ ]:
# Grafico a barre: confronto metriche Maschi vs Femmine
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metrics_to_plot))
width = 0.35

bars_m = ax.bar(x - width/2, mlp_results.loc['Maschi', metrics_to_plot],
                width, label='Maschi', color=COLOR_M, alpha=0.85)
bars_f = ax.bar(x + width/2, mlp_results.loc['Femmine', metrics_to_plot],
                width, label='Femmine', color=COLOR_F, alpha=0.85)

ax.set_ylabel('Valore')
ax.set_title('Mini-MLP — Confronto Maschi vs Femmine')
ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot, rotation=15)
ax.legend()
ax.set_ylim(0, 1)

for bar in bars_m:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)
for bar in bars_f:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

---

## Sezione 5: Confronto Modelli — Performance Gap

### Un modello piu' complesso risolve automaticamente il bias?

Abbiamo addestrato 3 modelli di complessita' crescente:
1. **Regressione Logistica** — modello lineare, il piu' semplice
2. **XGBoost** — ensemble di alberi, molto potente
3. **Mini-MLP** — rete neurale, deep learning

Confrontiamo ora le performance **per genere** tra tutti e tre i modelli.

In [ ]:
# Tabella combinata di tutti e 3 i modelli
all_models_results = pd.concat(
    [lr_results, xgb_results, mlp_results],
    keys=['Logistic Regression', 'XGBoost', 'Mini-MLP'],
    names=['Modello', 'Sottogruppo']
)
print('=== CONFRONTO COMPLETO — Tutti i Modelli ===')
print()
all_models_results

In [ ]:
# Grafico: AUC-ROC per modello e genere
models = ['Logistic Regression', 'XGBoost', 'Mini-MLP']
auc_m = [lr_results.loc['Maschi', 'AUC-ROC'], xgb_results.loc['Maschi', 'AUC-ROC'], mlp_results.loc['Maschi', 'AUC-ROC']]
auc_f = [lr_results.loc['Femmine', 'AUC-ROC'], xgb_results.loc['Femmine', 'AUC-ROC'], mlp_results.loc['Femmine', 'AUC-ROC']]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(models))
width = 0.35

bars_m = ax.bar(x - width/2, auc_m, width, label='Maschi', color=COLOR_M, alpha=0.85)
bars_f = ax.bar(x + width/2, auc_f, width, label='Femmine', color=COLOR_F, alpha=0.85)

ax.set_ylabel('AUC-ROC')
ax.set_title('AUC-ROC per Modello e Genere')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.set_ylim(0.5, 1.0)

for bar in bars_m:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=9)
for bar in bars_f:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Grafico: Sensitivity per modello e genere
sens_m = [lr_results.loc['Maschi', 'Recall (Sensitivity)'], xgb_results.loc['Maschi', 'Recall (Sensitivity)'], mlp_results.loc['Maschi', 'Recall (Sensitivity)']]
sens_f = [lr_results.loc['Femmine', 'Recall (Sensitivity)'], xgb_results.loc['Femmine', 'Recall (Sensitivity)'], mlp_results.loc['Femmine', 'Recall (Sensitivity)']]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(models))

bars_m = ax.bar(x - width/2, sens_m, width, label='Maschi', color=COLOR_M, alpha=0.85)
bars_f = ax.bar(x + width/2, sens_f, width, label='Femmine', color=COLOR_F, alpha=0.85)

ax.set_ylabel('Sensitivity (Recall)')
ax.set_title('Sensitivity per Modello e Genere')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.set_ylim(0, 1.0)

for bar in bars_m:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars_f:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Grafico: Specificity per modello e genere
spec_m = [lr_results.loc['Maschi', 'Specificity'], xgb_results.loc['Maschi', 'Specificity'], mlp_results.loc['Maschi', 'Specificity']]
spec_f = [lr_results.loc['Femmine', 'Specificity'], xgb_results.loc['Femmine', 'Specificity'], mlp_results.loc['Femmine', 'Specificity']]

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(models))

bars_m = ax.bar(x - width/2, spec_m, width, label='Maschi', color=COLOR_M, alpha=0.85)
bars_f = ax.bar(x + width/2, spec_f, width, label='Femmine', color=COLOR_F, alpha=0.85)

ax.set_ylabel('Specificity')
ax.set_title('Specificity per Modello e Genere')
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.legend()
ax.set_ylim(0.5, 1.0)

for bar in bars_m:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars_f:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

### Messaggio Chiave

**Il gap di performance tra maschi e femmine persiste INDIPENDENTEMENTE dalla complessita' del modello.**

Questo ci dice che il problema **non e' nell'algoritmo** ma nei **dati** che gli forniamo.
Un modello piu' potente non elimina il bias: lo riproduce fedelmente.

---

---

## Sezione 6: Performance per Etnia e Eta'

Estendiamo l'analisi del gap di performance ad **altri sottogruppi**: etnia e fasce di eta'.  
Usiamo il modello XGBoost come riferimento (il piu' performante tra quelli classici).

In [ ]:
# Analisi per Etnia: Caucasian vs Non-Caucasian
mask_cauc = test_features['ethnicity_Caucasian'] == 1
mask_non_cauc = test_features['ethnicity_Caucasian'] == 0

def evaluate_subgroup(model, X, y, mask, name):
    """Valuta un modello sklearn/xgboost su un sottogruppo."""
    X_sub = X[mask]
    y_sub = y[mask]
    y_pred = model.predict(X_sub)
    y_prob = model.predict_proba(X_sub)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_sub, y_pred).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    return {
        'Sottogruppo': name,
        'N campioni': len(y_sub),
        'Accuracy': round(accuracy_score(y_sub, y_pred), 4),
        'Precision': round(precision_score(y_sub, y_pred, zero_division=0), 4),
        'Sensitivity': round(recall_score(y_sub, y_pred, zero_division=0), 4),
        'Specificity': round(specificity, 4),
        'F1': round(f1_score(y_sub, y_pred, zero_division=0), 4),
        'AUC-ROC': round(roc_auc_score(y_sub, y_prob), 4)
    }

eth_results = pd.DataFrame([
    evaluate_subgroup(xgb_model, test_features, y_test, mask_cauc, 'Caucasian'),
    evaluate_subgroup(xgb_model, test_features, y_test, mask_non_cauc, 'Non-Caucasian')
]).set_index('Sottogruppo')

print('=== XGBoost — Performance per Etnia ===')
print()
eth_results

In [ ]:
# Grafico etnia
metrics_eth = ['AUC-ROC', 'Sensitivity', 'Specificity', 'F1']

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metrics_eth))
width = 0.35

bars1 = ax.bar(x - width/2, eth_results.loc['Caucasian', metrics_eth],
               width, label='Caucasian', color='#3498db', alpha=0.85)
bars2 = ax.bar(x + width/2, eth_results.loc['Non-Caucasian', metrics_eth],
               width, label='Non-Caucasian', color='#e67e22', alpha=0.85)

ax.set_ylabel('Valore')
ax.set_title('XGBoost — Performance per Etnia')
ax.set_xticks(x)
ax.set_xticklabels(metrics_eth)
ax.legend()
ax.set_ylim(0, 1)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Analisi per Eta': sotto/sopra la mediana (standardizzata -> split a 0)
mask_young = test_features['age'] <= 0  # sotto la mediana
mask_old = test_features['age'] > 0     # sopra la mediana

age_results = pd.DataFrame([
    evaluate_subgroup(xgb_model, test_features, y_test, mask_young, 'Eta <= mediana'),
    evaluate_subgroup(xgb_model, test_features, y_test, mask_old, 'Eta > mediana')
]).set_index('Sottogruppo')

print('=== XGBoost — Performance per Fascia di Eta ===')
print()
age_results

In [ ]:
# Grafico eta'
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(metrics_eth))
width = 0.35

bars1 = ax.bar(x - width/2, age_results.loc['Eta <= mediana', metrics_eth],
               width, label='Eta <= mediana', color='#27ae60', alpha=0.85)
bars2 = ax.bar(x + width/2, age_results.loc['Eta > mediana', metrics_eth],
               width, label='Eta > mediana', color='#8e44ad', alpha=0.85)

ax.set_ylabel('Valore')
ax.set_title('XGBoost — Performance per Fascia di Eta')
ax.set_xticks(x)
ax.set_xticklabels(metrics_eth)
ax.legend()
ax.set_ylim(0, 1)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Analisi Intersezionale: Genere x Etnia (4 gruppi)
mask_M_cauc = (test_features['gender_M'] == 1) & (test_features['ethnicity_Caucasian'] == 1)
mask_M_non_cauc = (test_features['gender_M'] == 1) & (test_features['ethnicity_Caucasian'] == 0)
mask_F_cauc = (test_features['gender_F'] == 1) & (test_features['ethnicity_Caucasian'] == 1)
mask_F_non_cauc = (test_features['gender_F'] == 1) & (test_features['ethnicity_Caucasian'] == 0)

intersect_results = pd.DataFrame([
    evaluate_subgroup(xgb_model, test_features, y_test, mask_M_cauc, 'M - Caucasian'),
    evaluate_subgroup(xgb_model, test_features, y_test, mask_M_non_cauc, 'M - Non-Caucasian'),
    evaluate_subgroup(xgb_model, test_features, y_test, mask_F_cauc, 'F - Caucasian'),
    evaluate_subgroup(xgb_model, test_features, y_test, mask_F_non_cauc, 'F - Non-Caucasian')
]).set_index('Sottogruppo')

print('=== XGBoost — Analisi Intersezionale (Genere x Etnia) ===')
print()
intersect_results

In [ ]:
# Grafico intersezionale
groups = ['M - Caucasian', 'M - Non-Caucasian', 'F - Caucasian', 'F - Non-Caucasian']
colors_intersect = [COLOR_M, '#5dade2', COLOR_F, '#f1948a']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, metric in enumerate(['AUC-ROC', 'Sensitivity', 'Specificity']):
    values = [intersect_results.loc[g, metric] for g in groups]
    bars = axes[idx].bar(range(len(groups)), values, color=colors_intersect, alpha=0.85)
    axes[idx].set_ylabel(metric)
    axes[idx].set_title(f'{metric} per Gruppo')
    axes[idx].set_xticks(range(len(groups)))
    axes[idx].set_xticklabels(groups, rotation=30, ha='right', fontsize=8)
    for bar in bars:
        axes[idx].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=8)

plt.suptitle('Analisi Intersezionale: Genere x Etnia', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## Sezione 7: FASE 4 — Il Bias Nasce dal Dataset

### L'esperimento fondamentale

Se il bias nasce dal dataset e non dall'algoritmo, allora **modificando il dataset** dovremmo vedere cambiare le performance.

**Esperimento**: rimuoviamo progressivamente donne dal training set e osserviamo cosa succede alle performance.

- **Scenario A**: Dataset originale (baseline)
- **Scenario B**: Rimozione del 20% delle donne dal training
- **Scenario C**: Rimozione del 40% delle donne dal training
- **Scenario D**: Rimozione del 60% delle donne dal training
- **Scenario E**: Rimozione del 80% delle donne dal training

**Importante**: il test set rimane SEMPRE lo stesso! Cambiamo solo i dati di addestramento.

In [ ]:
# Esperimento: rimozione progressiva delle donne dal training set
removal_percentages = [0, 20, 40, 60, 80]
scenario_names = ['A (0%)', 'B (20%)', 'C (40%)', 'D (60%)', 'E (80%)']

# Identifichiamo le donne nel training set
mask_F_train = train_features['gender_F'] == 1
mask_M_train = train_features['gender_M'] == 1

idx_F_train = train_features[mask_F_train].index
idx_M_train = train_features[mask_M_train].index

print(f'Donne nel training set: {len(idx_F_train)}')
print(f'Maschi nel training set: {len(idx_M_train)}')
print()

# Risultati per ogni scenario
female_removal_results = []

for pct, name in zip(removal_percentages, scenario_names):
    print(f'--- Scenario {name}: rimozione {pct}% delle donne ---')
    
    if pct == 0:
        # Dataset originale
        X_train_mod = train_features
        y_train_mod = y_train
    else:
        # Rimuoviamo una percentuale casuale di donne
        n_remove = int(len(idx_F_train) * pct / 100)
        np.random.seed(42)
        idx_remove = np.random.choice(idx_F_train, size=n_remove, replace=False)
        idx_keep = train_features.index.difference(idx_remove)
        X_train_mod = train_features.loc[idx_keep]
        y_train_mod = y_train[idx_keep]
    
    n_f = (X_train_mod['gender_F'] == 1).sum()
    n_m = (X_train_mod['gender_M'] == 1).sum()
    print(f'  Training: {len(X_train_mod)} righe (M: {n_m}, F: {n_f})')
    
    # Addestramento XGBoost con stessi iperparametri
    xgb_temp = XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        random_state=42, eval_metric='logloss'
    )
    xgb_temp.fit(X_train_mod, y_train_mod)
    
    # Valutazione sul test set (INVARIATO)
    res = evaluate_model_subgroups(xgb_temp, test_features, y_test, name)
    
    female_removal_results.append({
        'Scenario': name,
        '% Donne rimosse': pct,
        'AUC Overall': res.loc['Overall', 'AUC-ROC'],
        'AUC Maschi': res.loc['Maschi', 'AUC-ROC'],
        'AUC Femmine': res.loc['Femmine', 'AUC-ROC'],
        'Sens. Overall': res.loc['Overall', 'Recall (Sensitivity)'],
        'Sens. Maschi': res.loc['Maschi', 'Recall (Sensitivity)'],
        'Sens. Femmine': res.loc['Femmine', 'Recall (Sensitivity)'],
        'Spec. Overall': res.loc['Overall', 'Specificity'],
        'Spec. Maschi': res.loc['Maschi', 'Specificity'],
        'Spec. Femmine': res.loc['Femmine', 'Specificity']
    })
    print(f'  AUC-ROC -> Overall: {res.loc["Overall", "AUC-ROC"]:.4f}, M: {res.loc["Maschi", "AUC-ROC"]:.4f}, F: {res.loc["Femmine", "AUC-ROC"]:.4f}')
    print()

df_fem_removal = pd.DataFrame(female_removal_results).set_index('Scenario')
print('=== Risultati Rimozione Progressiva Donne ===')
print()
df_fem_removal

In [ ]:
# Grafico: AUC-ROC al variare della % di donne rimosse
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(removal_percentages, df_fem_removal['AUC Overall'], 'o-',
        color=COLOR_OVERALL, linewidth=2, markersize=8, label='Overall')
ax.plot(removal_percentages, df_fem_removal['AUC Maschi'], 's-',
        color=COLOR_M, linewidth=2, markersize=8, label='Maschi')
ax.plot(removal_percentages, df_fem_removal['AUC Femmine'], '^-',
        color=COLOR_F, linewidth=2, markersize=8, label='Femmine')

ax.set_xlabel('% Donne rimosse dal Training Set')
ax.set_ylabel('AUC-ROC')
ax.set_title('Effetto della Rimozione delle Donne sulla AUC-ROC')
ax.legend()
ax.set_xticks(removal_percentages)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Grafico: Sensitivity al variare della % di donne rimosse
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(removal_percentages, df_fem_removal['Sens. Overall'], 'o-',
        color=COLOR_OVERALL, linewidth=2, markersize=8, label='Overall')
ax.plot(removal_percentages, df_fem_removal['Sens. Maschi'], 's-',
        color=COLOR_M, linewidth=2, markersize=8, label='Maschi')
ax.plot(removal_percentages, df_fem_removal['Sens. Femmine'], '^-',
        color=COLOR_F, linewidth=2, markersize=8, label='Femmine')

ax.set_xlabel('% Donne rimosse dal Training Set')
ax.set_ylabel('Sensitivity')
ax.set_title('Effetto della Rimozione delle Donne sulla Sensitivity')
ax.legend()
ax.set_xticks(removal_percentages)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Grafico: Specificity al variare della % di donne rimosse
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(removal_percentages, df_fem_removal['Spec. Overall'], 'o-',
        color=COLOR_OVERALL, linewidth=2, markersize=8, label='Overall')
ax.plot(removal_percentages, df_fem_removal['Spec. Maschi'], 's-',
        color=COLOR_M, linewidth=2, markersize=8, label='Maschi')
ax.plot(removal_percentages, df_fem_removal['Spec. Femmine'], '^-',
        color=COLOR_F, linewidth=2, markersize=8, label='Femmine')

ax.set_xlabel('% Donne rimosse dal Training Set')
ax.set_ylabel('Specificity')
ax.set_title('Effetto della Rimozione delle Donne sulla Specificity')
ax.legend()
ax.set_xticks(removal_percentages)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Sezione 8: FASE 4 (cont.) — Scenario Inverso

Per dimostrare che l'effetto e' **simmetrico**, ripetiamo l'esperimento ma questa volta
**rimuovendo progressivamente i maschi** dal training set.

Se il bias dipende dal dataset, allora rimuovere maschi dovrebbe peggiorare le performance sui maschi.

In [ ]:
# Esperimento inverso: rimozione progressiva dei maschi dal training set
male_removal_results = []

for pct, name in zip(removal_percentages, scenario_names):
    print(f'--- Scenario {name}: rimozione {pct}% dei maschi ---')
    
    if pct == 0:
        X_train_mod = train_features
        y_train_mod = y_train
    else:
        n_remove = int(len(idx_M_train) * pct / 100)
        np.random.seed(42)
        idx_remove = np.random.choice(idx_M_train, size=n_remove, replace=False)
        idx_keep = train_features.index.difference(idx_remove)
        X_train_mod = train_features.loc[idx_keep]
        y_train_mod = y_train[idx_keep]
    
    n_f = (X_train_mod['gender_F'] == 1).sum()
    n_m = (X_train_mod['gender_M'] == 1).sum()
    print(f'  Training: {len(X_train_mod)} righe (M: {n_m}, F: {n_f})')
    
    xgb_temp = XGBClassifier(
        n_estimators=100, max_depth=6, learning_rate=0.1,
        random_state=42, eval_metric='logloss'
    )
    xgb_temp.fit(X_train_mod, y_train_mod)
    
    res = evaluate_model_subgroups(xgb_temp, test_features, y_test, name)
    
    male_removal_results.append({
        'Scenario': name,
        '% Maschi rimossi': pct,
        'AUC Overall': res.loc['Overall', 'AUC-ROC'],
        'AUC Maschi': res.loc['Maschi', 'AUC-ROC'],
        'AUC Femmine': res.loc['Femmine', 'AUC-ROC'],
        'Sens. Overall': res.loc['Overall', 'Recall (Sensitivity)'],
        'Sens. Maschi': res.loc['Maschi', 'Recall (Sensitivity)'],
        'Sens. Femmine': res.loc['Femmine', 'Recall (Sensitivity)'],
        'Spec. Overall': res.loc['Overall', 'Specificity'],
        'Spec. Maschi': res.loc['Maschi', 'Specificity'],
        'Spec. Femmine': res.loc['Femmine', 'Specificity']
    })
    print(f'  AUC-ROC -> Overall: {res.loc["Overall", "AUC-ROC"]:.4f}, M: {res.loc["Maschi", "AUC-ROC"]:.4f}, F: {res.loc["Femmine", "AUC-ROC"]:.4f}')
    print()

df_male_removal = pd.DataFrame(male_removal_results).set_index('Scenario')
print('=== Risultati Rimozione Progressiva Maschi ===')
print()
df_male_removal

In [ ]:
# Grafici per la rimozione dei maschi
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, (metric_prefix, title) in enumerate([
    ('AUC', 'AUC-ROC'), ('Sens.', 'Sensitivity'), ('Spec.', 'Specificity')
]):
    ax = axes[idx]
    ax.plot(removal_percentages, df_male_removal[f'{metric_prefix} Overall'], 'o-',
            color=COLOR_OVERALL, linewidth=2, markersize=8, label='Overall')
    ax.plot(removal_percentages, df_male_removal[f'{metric_prefix} Maschi'], 's-',
            color=COLOR_M, linewidth=2, markersize=8, label='Maschi')
    ax.plot(removal_percentages, df_male_removal[f'{metric_prefix} Femmine'], '^-',
            color=COLOR_F, linewidth=2, markersize=8, label='Femmine')
    ax.set_xlabel('% Maschi rimossi dal Training Set')
    ax.set_ylabel(title)
    ax.set_title(f'{title} — Rimozione Maschi')
    ax.legend()
    ax.set_xticks(removal_percentages)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Confronto diretto: AUC del gruppo rimosso nei due esperimenti
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(removal_percentages, df_fem_removal['AUC Femmine'], '^-',
        color=COLOR_F, linewidth=2, markersize=8, label='AUC Femmine (rimuovendo donne)')
ax.plot(removal_percentages, df_male_removal['AUC Maschi'], 's-',
        color=COLOR_M, linewidth=2, markersize=8, label='AUC Maschi (rimuovendo maschi)')

ax.set_xlabel('% del Gruppo rimossa dal Training Set')
ax.set_ylabel('AUC-ROC')
ax.set_title('Simmetria del Bias: Rimuovere un Gruppo Penalizza Quel Gruppo')
ax.legend()
ax.set_xticks(removal_percentages)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Osservazione

L'effetto e' **simmetrico**: rimuovere campioni di un qualsiasi gruppo dal training set penalizza le performance su quel gruppo nel test set.  
Questo conferma che il bias e' una proprieta' dei **dati**, non dell'algoritmo.

---

---

## Sezione 9: FASE 4 (cont.) — Riequilibrio del Dataset

Se il problema e' nei dati, possiamo provare a **riequilibrarli**?

Creiamo un training set **bilanciato** per genere: riduciamo i maschi al numero delle femmine (undersampling).
Poi confrontiamo le performance con il modello addestrato sul dataset originale.

In [ ]:
# Creazione del training set bilanciato per genere
n_females = len(idx_F_train)
n_males = len(idx_M_train)

print(f'Dataset originale: {n_males} maschi, {n_females} femmine')
print(f'Rapporto M/F: {n_males/n_females:.2f}')
print()

# Undersampling dei maschi
np.random.seed(42)
idx_M_sampled = np.random.choice(idx_M_train, size=n_females, replace=False)
idx_balanced = np.concatenate([idx_M_sampled, idx_F_train.values])
np.random.shuffle(idx_balanced)

X_train_balanced = train_features.loc[idx_balanced]
y_train_balanced = y_train[idx_balanced]

n_f_bal = (X_train_balanced['gender_F'] == 1).sum()
n_m_bal = (X_train_balanced['gender_M'] == 1).sum()
print(f'Dataset bilanciato: {n_m_bal} maschi, {n_f_bal} femmine')
print(f'Totale: {len(X_train_balanced)} righe')

In [ ]:
# Addestramento XGBoost sul dataset bilanciato
xgb_balanced = XGBClassifier(
    n_estimators=100, max_depth=6, learning_rate=0.1,
    random_state=42, eval_metric='logloss'
)
xgb_balanced.fit(X_train_balanced, y_train_balanced)

# Valutazione
balanced_results = evaluate_model_subgroups(xgb_balanced, test_features, y_test, 'XGBoost Bilanciato')
print('=== XGBoost su Dataset Bilanciato ===')
print()
balanced_results

In [ ]:
# Confronto: Dataset Originale vs Dataset Bilanciato
comparison_balance = pd.concat(
    [xgb_results, balanced_results],
    keys=['XGBoost (Originale)', 'XGBoost (Bilanciato)'],
    names=['Modello', 'Sottogruppo']
)
print('=== CONFRONTO: Originale vs Bilanciato ===')
print()
comparison_balance

In [ ]:
# Grafico confronto Originale vs Bilanciato
metrics_comp = ['AUC-ROC', 'Recall (Sensitivity)', 'Specificity']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, metric in enumerate(metrics_comp):
    ax = axes[idx]
    
    # Originale
    val_m_orig = xgb_results.loc['Maschi', metric]
    val_f_orig = xgb_results.loc['Femmine', metric]
    
    # Bilanciato
    val_m_bal = balanced_results.loc['Maschi', metric]
    val_f_bal = balanced_results.loc['Femmine', metric]
    
    x = np.arange(2)
    width = 0.35
    
    bars_m = ax.bar(x - width/2, [val_m_orig, val_m_bal], width,
                    label='Maschi', color=COLOR_M, alpha=0.85)
    bars_f = ax.bar(x + width/2, [val_f_orig, val_f_bal], width,
                    label='Femmine', color=COLOR_F, alpha=0.85)
    
    ax.set_ylabel(metric)
    ax.set_title(metric)
    ax.set_xticks(x)
    ax.set_xticklabels(['Originale', 'Bilanciato'])
    ax.legend()
    
    for bar in bars_m:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
    for bar in bars_f:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('Effetto del Riequilibrio del Dataset', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---

## Sezione 10: Riepilogo e Discussione

### Tabella Riassuntiva di Tutti gli Esperimenti

In [ ]:
# Tabella riassuntiva finale
summary_data = []

# 3 modelli base
for model_name, res in [('Logistic Regression', lr_results), ('XGBoost', xgb_results), ('Mini-MLP', mlp_results)]:
    summary_data.append({
        'Esperimento': model_name,
        'AUC M': res.loc['Maschi', 'AUC-ROC'],
        'AUC F': res.loc['Femmine', 'AUC-ROC'],
        'Gap AUC (M-F)': round(res.loc['Maschi', 'AUC-ROC'] - res.loc['Femmine', 'AUC-ROC'], 4),
        'Sens. M': res.loc['Maschi', 'Recall (Sensitivity)'],
        'Sens. F': res.loc['Femmine', 'Recall (Sensitivity)'],
        'Gap Sens. (M-F)': round(res.loc['Maschi', 'Recall (Sensitivity)'] - res.loc['Femmine', 'Recall (Sensitivity)'], 4)
    })

# Scenari rimozione donne
for _, row in df_fem_removal.iterrows():
    pct = row['% Donne rimosse']
    if pct > 0:
        summary_data.append({
            'Esperimento': f'XGB - {int(pct)}% donne rimosse',
            'AUC M': row['AUC Maschi'],
            'AUC F': row['AUC Femmine'],
            'Gap AUC (M-F)': round(row['AUC Maschi'] - row['AUC Femmine'], 4),
            'Sens. M': row['Sens. Maschi'],
            'Sens. F': row['Sens. Femmine'],
            'Gap Sens. (M-F)': round(row['Sens. Maschi'] - row['Sens. Femmine'], 4)
        })

# Dataset bilanciato
summary_data.append({
    'Esperimento': 'XGB - Dataset Bilanciato',
    'AUC M': balanced_results.loc['Maschi', 'AUC-ROC'],
    'AUC F': balanced_results.loc['Femmine', 'AUC-ROC'],
    'Gap AUC (M-F)': round(balanced_results.loc['Maschi', 'AUC-ROC'] - balanced_results.loc['Femmine', 'AUC-ROC'], 4),
    'Sens. M': balanced_results.loc['Maschi', 'Recall (Sensitivity)'],
    'Sens. F': balanced_results.loc['Femmine', 'Recall (Sensitivity)'],
    'Gap Sens. (M-F)': round(balanced_results.loc['Maschi', 'Recall (Sensitivity)'] - balanced_results.loc['Femmine', 'Recall (Sensitivity)'], 4)
})

df_summary = pd.DataFrame(summary_data).set_index('Esperimento')
print('=== TABELLA RIASSUNTIVA FINALE ===')
print()
df_summary

### Messaggi Chiave

---

**1. Il performance gap per genere esiste in TUTTI i modelli**  
Regressione Logistica, XGBoost e rete neurale mostrano tutti differenze di performance tra maschi e femmine.

**2. Un modello piu' potente NON elimina il bias**  
Passare da un modello lineare a un modello di deep learning non risolve il problema. Anzi, modelli piu' potenti possono amplificare i pattern (anche quelli indesiderati) presenti nei dati.

**3. Rimuovere donne dal training peggiora la performance sulle donne**  
Meno dati su un sottogruppo = peggiori predizioni per quel sottogruppo. L'effetto e' simmetrico.

**4. Riequilibrare il dataset riduce il gap**  
Anche un semplice undersampling (ridurre i maschi al livello delle femmine) puo' migliorare l'equita' del modello.

**5. Il bias nasce dal DATASET, non dall'algoritmo**  
L'algoritmo impara cio' che trova nei dati. Se i dati sono sbilanciati o non rappresentativi, le predizioni saranno inique.

---

### Domande per la Discussione

1. Nella vostra pratica clinica, avete mai osservato differenze di trattamento legate al genere o all'etnia?

2. Se un modello predittivo funziona meglio per i maschi, che impatto potrebbe avere sulle pazienti donne?

3. Chi e' responsabile di garantire che un modello di AI clinica sia equo? I data scientist? I medici? Le istituzioni?

4. Bastano le soluzioni tecniche (come il riequilibrio del dataset) o servono anche interventi organizzativi e culturali?

5. Come possiamo raccogliere dati clinici piu' rappresentativi in futuro?

---

*Fine del notebook — Fase 3 e 4 del Workshop*